[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/describing_data_course.ipynb)

# Descriptive statistics full course: the practice set, in Python

Thirty-five exercises in four tiers, the same as the practice spreadsheet: **tier 1** warm-ups (one idea, one step),
**tier 2** core problems (several steps), **tier 3** stretch problems (combine ideas, spot the trap, work backwards)
and **tier 4**, the case: all 480 orders of Wrenfield Home, an online homeware store (a fictional company; our own
data), and the question of which delivery carrier it should keep. Each tier is harder than the last.

Under every tier there's a short **how you'd do this in Python** cell: the same calculations with Python's
`statistics` module, `numpy` and `pandas`, so you can use them on your own data. And a few cells you can move
yourself: a slider that drags one shipment time while you watch the mean and the median, a dot plot with its ±s band,
a seventh day that pulls a correlation down, and the case's histogram with a class-width slider.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no
install) and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Numbers without
units and without thousands separators: `120000`, not `120,000`. Percents as the number: `5.5` for 5.5%. Dollars
without the sign: `22.24`. You can type the arithmetic too: `22/400*100`. For several answers, separate them with
commas, in the order the question lists them (the box's label shows the order). Words as words: `nominal`, `median`,
`yes`. You can also put your answer between the quotes in a cell's last line, `practice("P1.4", "")`, and run the
cell: it checks straight away.

**The cheat sheet.** Every formula and spreadsheet function from the course on two pages:
[descriptive statistics cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/descriptive_statistics_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import ast, math, operator, re, random, textwrap
import statistics as st
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interact, IntSlider, FloatSlider, Text, Button, Output, HBox, VBox, Layout, HTML

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # data in chalk, chalk smudge, ghost lines
SKY, VIOLET, CORAL, GOLD, MINT = "#6fb6bb", "#b49ad6", "#e3896b", "#e8c56b", "#a0c27c"
# centre SKY · spread VIOLET · flagged CORAL · the one key mark GOLD; the two carriers: Corvey SKY, Tamsby MINT
PAIR = {"Corvey": SKY, "Tamsby": MINT}

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="our own data · computed"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

def dot_plot(ax, xs, y=0.0, color=CHALK, step=0.18, size=70):
    """Equal values stack upwards, as in the course's dot plots."""
    seen = {}
    for x in xs:
        k = seen.get(x, 0); seen[x] = k + 1
        ax.scatter([x], [y + k * step], s=size, color=color, zorder=3)

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}

def calc(text):
    """Plain arithmetic only: numbers, + − × ÷, powers, brackets and sqrt(…)."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") == "sqrt" and len(n.args) == 1: return math.sqrt(ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNITS = r"\b(days?|hours?|minutes?|mins?|ml|degrees?|orders?|times)\b|[$%°]"
SAME = {"machine two": {"machine 2", "two", "2", "second", "the second", "machine two"},
        "inferential": {"inference", "inferential", "inferential statistics"},
        "positive": {"positive", "+", "pos"}, "no": {"no", "n"}, "yes": {"yes", "y"}}

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    t = re.sub(UNITS, "", s, flags=re.I).strip()
    try: return calc(t)
    except Exception: return " ".join(s.lower().split())

def right(got, want, tol):
    if isinstance(want, str):
        g = got if isinstance(got, str) else f"{got:g}"
        return g == want.lower() or g in SAME.get(want.lower(), ())
    return not isinstance(got, str) and abs(got - want) <= max(tol, 1e-6)

def show(v):
    if isinstance(v, str): return v
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the same questions and answers as the spreadsheet) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'Data and statistics',
     "q": "A customer file records each customer's country of residence. Which scale of measurement is that variable: nominal, ordinal, interval or ratio?",
     "parts": ['type one word: nominal, ordinal, interval or ratio'],
     "answer": ['nominal'], "tol": 0, "function": '—',
     "working": 'A country is a label with no natural order, so it is nominal.'},
    {"id": 'P1.2', "tier": 1, "group": 'Tables and charts for one variable',
     "q": 'Of 50 support calls, 18 were about billing. What is the relative frequency of billing calls?',
     "parts": ['relative frequency (a decimal)'],
     "answer": [0.36], "tol": 0, "function": 'COUNTIF, then divide by the total',
     "working": '18 / 50 = 0.36.'},
    {"id": 'P1.3', "tier": 1, "group": 'Tables and charts for one variable',
     "q": 'In a pie chart, what angle (in degrees) does a category with 25% of the data get?',
     "parts": ['angle (degrees)'],
     "answer": [90.0], "tol": 0, "function": '—',
     "working": '0.25 × 360° = 90°.'},
    {"id": 'P1.4', "tier": 1, "group": 'Two variables',
     "q": 'An app had 400 sessions and 22 of them ended in a purchase. What is the conversion rate, as a percent?',
     "parts": ['conversion rate (%)'],
     "answer": [5.5], "tol": 0, "function": 'COUNTIFS, then divide',
     "working": '22 / 400 = 0.055 = 5.5%.'},
    {"id": 'P1.5', "tier": 1, "group": 'Measures of location',
     "q": 'Five order values ($): 42, 58, 35, 61, 54. What is the mean?',
     "parts": ['mean ($)'],
     "answer": [50.0], "tol": 0, "function": 'AVERAGE',
     "working": 'Sum 250; 250 / 5 = 50.'},
    {"id": 'P1.6', "tier": 1, "group": 'Measures of location',
     "q": 'Seven lead times (days): 4, 9, 6, 5, 12, 7, 6. What is the median?',
     "parts": ['median (days)'],
     "answer": [6.0], "tol": 0, "function": 'MEDIAN',
     "working": 'Sorted: 4 5 6 6 7 9 12; the 4th value is 6.'},
    {"id": 'P1.7', "tier": 1, "group": 'Measures of variability',
     "q": 'Daily sales ($ thousands): 120, 95, 143, 110, 132. What is the range?',
     "parts": ['range ($ thousands)'],
     "answer": [48], "tol": 0, "function": 'MAX, MIN',
     "working": '143 − 95 = 48.'},
    {"id": 'P1.8', "tier": 1, "group": 'Shape, z-scores, outliers and boxplots',
     "q": "A store's weekly sales were $82k in a region where the mean is $70k and the standard deviation $8k. What is the z-score?",
     "parts": ['z-score'],
     "answer": [1.5], "tol": 0, "function": 'STANDARDIZE',
     "working": '(82 − 70) / 8 = 1.5.'},
    {"id": 'P1.9', "tier": 1, "group": 'Covariance and correlation',
     "q": 'Across twelve months, months with more advertising spend also tended to have higher sales. Is the covariance of spend and sales positive, negative or zero?',
     "parts": ['type one word: positive, negative or zero'],
     "answer": ['positive'], "tol": 0, "function": 'COVARIANCE.S',
     "working": 'Above-average spend pairs with above-average sales, so the products of deviations are mostly positive.'},
    {"id": 'P2.1', "tier": 2, "group": 'Data and statistics',
     "q": 'A bank surveys 800 of its 120,000 small-business clients; 296 say they plan to borrow next year. (a) What percent of the sample plan to borrow? (b) Is "about 37% of all clients plan to borrow" descriptive or inferential?',
     "parts": ['(a) percent of the sample (%)', '(b) type one word: descriptive or inferential'],
     "answer": [37.0, 'inferential'], "tol": 0, "function": 'COUNTIF, then divide',
     "working": '(a) 296 / 800 = 0.37 = 37%. (b) It estimates the whole population from a sample, so it is inference (valid if the 800 were chosen at random).'},
    {"id": 'P2.2', "tier": 2, "group": 'Tables and charts for one variable',
     "q": '24 invoices took these times to process (minutes): 12, 17, 22, 14, 19, 26, 31, 18, 15, 23, 20, 16, 13, 27, 21, 18, 34, 16, 19, 24, 11, 17, 22, 14. Using classes 10–14, 15–19, 20–24, 25–29 and 30–34, give the five class frequencies, then the percent of invoices done in under 25 minutes.',
     "parts": ['frequency, 10–14', 'frequency, 15–19', 'frequency, 20–24', 'frequency, 25–29', 'frequency, 30–34', 'percent under 25 minutes (%)'],
     "answer": [5, 9, 6, 2, 2, 83.33333333333333], "tol": 0.1, "function": 'FREQUENCY (upper limits 14, 19, 24, 29, 34)',
     "working": 'Count each class: 5, 9, 6, 2, 2 (sum 24). Under 25 minutes: 5 + 9 + 6 = 20; 20 / 24 ≈ 83.3%.'},
    {"id": 'P2.3', "tier": 2, "group": 'Two variables',
     "q": "Two warehouses: North shipped 1,200 orders and 30 arrived damaged; South shipped 800 and 28 arrived damaged. Give each warehouse's damage rate as a row percent.",
     "parts": ['North damage rate (%)', 'South damage rate (%)'],
     "answer": [2.5, 3.5], "tol": 0, "function": "COUNTIFS, then divide by each row's total",
     "working": '30 / 1200 = 2.5%; 28 / 800 = 3.5%. South has fewer damaged orders in total but the higher rate.'},
    {"id": 'P2.4', "tier": 2, "group": 'Measures of location',
     "q": 'A shop sold 120 units at $15, 80 units at $22 and 50 units at $40. What was the average price per unit sold?',
     "parts": ['average price per unit ($)'],
     "answer": [22.24], "tol": 0, "function": 'SUMPRODUCT, SUM',
     "working": '(120 × 15 + 80 × 22 + 50 × 40) / 250 = (1800 + 1760 + 2000) / 250 = 5560 / 250 = $22.24.'},
    {"id": 'P2.5', "tier": 2, "group": 'Measures of location',
     "q": 'Sales grew 8%, then 12%, then fell 5%. What is the average annual growth rate, as a percent (to two decimals)?',
     "parts": ['average annual growth (%, two decimals)'],
     "answer": [4.74], "tol": 0.01, "function": 'GEOMEAN',
     "working": 'Factors 1.08 × 1.12 × 0.95 = 1.14912; the cube root ≈ 1.0474, so about 4.74% a year.'},
    {"id": 'P2.6', "tier": 2, "group": 'Measures of location',
     "q": 'Fifteen call durations (minutes), sorted: 3, 5, 6, 7, 8, 8, 9, 10, 11, 12, 13, 15, 18, 21, 30. Find Q1, the median and Q3 with the (n + 1) rule.',
     "parts": ['Q1', 'median', 'Q3'],
     "answer": [7.0, 10.0, 15.0], "tol": 0, "function": 'QUARTILE.EXC, MEDIAN',
     "working": 'Positions 0.25 × 16 = 4, 0.5 × 16 = 8, 0.75 × 16 = 12: Q1 = 7, median = 10, Q3 = 15.'},
    {"id": 'P2.7', "tier": 2, "group": 'Measures of variability',
     "q": "A fund's weekly returns (%): 1.2, −0.4, 0.8, 2.0, −1.1, 0.5. Find the sample variance and the sample standard deviation (two decimals).",
     "parts": ['sample variance', 'sample standard deviation (two decimals)'],
     "answer": [1.24, 1.11], "tol": 0.01, "function": 'VAR.S, STDEV.S',
     "working": 'Mean 3.0 / 6 = 0.5. Deviations 0.7, −0.9, 0.3, 1.5, −1.6, 0; squares sum to 6.2; s² = 6.2 / 5 = 1.24; s = √1.24 ≈ 1.11.'},
    {"id": 'P2.8', "tier": 2, "group": 'Measures of variability',
     "q": 'Machine one fills 250 ml bottles with a standard deviation of 3 ml; machine two fills 1,000 ml bottles with a standard deviation of 8 ml. Give each coefficient of variation (%), and say which is more consistent.',
     "parts": ['CV, machine one (%)', 'CV, machine two (%)', 'more consistent: machine one or machine two'],
     "answer": [1.2, 0.8, 'machine two'], "tol": 0, "function": 'STDEV.S, AVERAGE',
     "working": 'CV = s / x̄ × 100%: 3 / 250 = 1.2%; 8 / 1000 = 0.8%. Machine two varies less relative to its mean.'},
    {"id": 'P2.9', "tier": 2, "group": 'Shape, z-scores, outliers and boxplots',
     "q": 'Eleven days of website sign-ups: 42, 38, 45, 51, 40, 44, 39, 47, 43, 95, 41. Give the five-number summary, the upper outlier limit (Q3 + 1.5 IQR), and any outliers.',
     "parts": ['minimum', 'Q1', 'median', 'Q3', 'maximum', 'upper outlier limit', 'the outlier'],
     "answer": [38.0, 40.0, 43.0, 47.0, 95.0, 57.5, 95], "tol": 0, "function": 'MIN, QUARTILE.EXC, MEDIAN, MAX',
     "working": 'Sorted: 38 39 40 41 42 43 44 45 47 51 95. Median the 6th: 43. Q1 at position 3: 40; Q3 at 9: 47. IQR 7; 1.5 × 7 = 10.5; limits 29.5 and 57.5; 95 is an outlier.'},
    {"id": 'P2.10', "tier": 2, "group": 'Covariance and correlation',
     "q": 'Five months of advertising spend ($k): 2, 3, 5, 6, 9, and sales ($k): 30, 34, 41, 45, 50. Find the sample covariance and the correlation (two decimals).',
     "parts": ['sample covariance', 'correlation r (two decimals)'],
     "answer": [21.75, 0.98], "tol": 0.01, "function": 'COVARIANCE.S, CORREL',
     "working": 'Means 5 and 40. Products of deviations: 30, 12, 0, 5, 40 = 87; s_xy = 87 / 4 = 21.75. r = 87 / √(30 × 262) ≈ 0.98.'},
    {"id": 'P3.1', "tier": 3, "group": 'Two variables',
     "q": "Two sales reps. Rhea closed 40 of 50 small deals and 15 of 50 large ones; Tomas closed 63 of 90 small and 2 of 10 large. Give each rep's overall close rate (%), and say who is better within each deal size.",
     "parts": ["Rhea's overall close rate (%)", "Tomas's overall close rate (%)", 'better in each deal size: Rhea or Tomas'],
     "answer": [55.0, 65.0, 'Rhea'], "tol": 0, "function": 'COUNTIFS, then divide',
     "working": "Overall: Rhea 55 / 100 = 55%, Tomas 65 / 100 = 65%. Small deals: 80% vs 70%; large: 30% vs 20%: Rhea wins both. Tomas looks better only because 90% of his deals were the easier small ones (Simpson's paradox)."},
    {"id": 'P3.2', "tier": 3, "group": 'Two variables',
     "q": 'A slide shows two satisfaction scores, 95 and 100, as bars on an axis that starts at 90. How many times as long does the second bar look, and what is the true ratio (two decimals)?',
     "parts": ['how many times as long it looks', 'the true ratio (two decimals)'],
     "answer": [2.0, 1.05], "tol": 0.01, "function": '—',
     "working": 'On the slide: (100 − 90) / (95 − 90) = 2, twice as long. Truly 100 / 95 ≈ 1.05: about 5% higher.'},
    {"id": 'P3.3', "tier": 3, "group": 'Measures of location',
     "q": 'Revenue fell 20% in one year and rose 25% the next. A report says growth averaged 2.5% a year. What was the true average annual growth rate (%)?',
     "parts": ['true average annual growth (%)'],
     "answer": [0.0], "tol": 0, "function": 'GEOMEAN',
     "working": 'Factors 0.8 × 1.25 = 1.0; the square root of 1 is 1, so 0% a year: revenue is back where it started. The arithmetic mean of the rates, 2.5%, is wrong for growth.'},
    {"id": 'P3.4', "tier": 3, "group": 'Measures of variability',
     "q": 'Five weekly order counts have a mean of 12. Four of them are 8, 10, 16 and 12. Find the fifth, then the sample standard deviation of all five (two decimals).',
     "parts": ['the fifth count', 'sample standard deviation of all five (two decimals)'],
     "answer": [14, 3.16], "tol": 0.01, "function": 'SUM, STDEV.S',
     "working": 'The five sum to 5 × 12 = 60, so the fifth is 60 − 46 = 14. Deviations −4, −2, 4, 0, 2; squares sum to 40; s² = 40 / 4 = 10; s = √10 ≈ 3.16.'},
    {"id": 'P3.5', "tier": 3, "group": 'Measures of variability',
     "q": 'A data set has a standard deviation of $4. What is the new standard deviation if (a) $5 is added to every value, (b) every value is multiplied by 1.1?',
     "parts": ['(a) after adding $5 ($)', '(b) after multiplying by 1.1 ($)'],
     "answer": [4.0, 4.4], "tol": 0.001, "function": 'STDEV.S',
     "working": "(a) Adding a constant moves every value and the mean together, so the deviations, and s, don't change: $4. (b) Multiplying scales every deviation by 1.1: s = 4 × 1.1 = $4.40."},
    {"id": 'P3.6', "tier": 3, "group": 'Shape, z-scores, outliers and boxplots',
     "q": 'Monthly invoice totals have a mean of $2,400 and a standard deviation of $300. (a) If the shape is unknown, at least what percent lie between $1,800 and $3,000? (b) If the distribution is bell-shaped, about what percent?',
     "parts": ['(a) at least this percent (%)', '(b) about this percent if bell-shaped (%)'],
     "answer": [75.0, 95.0], "tol": 0, "function": '—',
     "working": '$1,800 and $3,000 are two standard deviations either side (z = ±2). (a) Chebyshev: at least 1 − 1/2² = 75%. (b) The empirical rule: about 95%.'},
    {"id": 'P3.7', "tier": 3, "group": 'Shape, z-scores, outliers and boxplots',
     "q": 'Ten delivery times (minutes): 20, 22, 23, 23, 24, 25, 25, 26, 27, 40. (a) The upper limit by the quartile rule (Q3 + 1.5 IQR, with the (n + 1) rule and interpolation). (b) The z-score of 40 (two decimals). Do the two outlier rules agree?',
     "parts": ['(a) upper limit, Q3 + 1.5 IQR', '(b) z-score of 40 (two decimals)'],
     "answer": [31.5, 2.64], "tol": 0.01, "function": 'QUARTILE.EXC, STANDARDIZE',
     "working": "Q1 at position 2.75: 22 + 0.75 × 1 = 22.75; Q3 at 8.25: 26 + 0.25 × 1 = 26.25; IQR 3.5; limit 26.25 + 5.25 = 31.5, so 40 is flagged. Mean 25.5, s ≈ 5.48, z ≈ 2.64 < 3, so the z rule doesn't flag it: the outlier itself inflates s. The rules disagree; look at the value."},
    {"id": 'P3.8', "tier": 3, "group": 'Covariance and correlation',
     "q": "At five prices ($8, $9, $10, $11, $12) a product's weekly profit was $30k, $45k, $50k, $45k and $30k. Find the correlation between price and profit. Does it mean price doesn't matter?",
     "parts": ['correlation r', "does it mean price doesn't matter? type one word: yes or no"],
     "answer": [0.0, 'no'], "tol": 0.001, "function": 'CORREL',
     "working": 'Products of deviations: (−2)(−10) + (−1)(5) + 0 + (1)(5) + (2)(−10) = 0, so r = 0. Price matters a lot, as an arch peaking at $10; r measures straight-line relationships only. Plot first.'},
    {"id": 'P4.1', "tier": 4, "group": 'Case study',
     "q": "Reproduce the video's first table: the percent of the 480 orders in each delivery zone (city, town, rural; two decimals).",
     "parts": ['city (%)', 'town (%)', 'rural (%)'],
     "answer": [51.25, 34.79, 13.96], "tol": 0.01, "function": 'COUNTIF',
     "working": 'COUNTIF the zone column: 246, 167, 67. Divide by 480: 51.25%, 34.79%, 13.96%.'},
    {"id": 'P4.2', "tier": 4, "group": 'Case study',
     "q": 'Order value: give the mean and the median (two decimals), and the share of orders below the mean (%, whole number). Which figure should the quarterly review call a typical order?',
     "parts": ['mean order value ($)', 'median order value ($)', 'orders below the mean (%, whole number)', 'the typical order: type one word: mean or median'],
     "answer": [73.77, 56.02, 63.0, 'median'], "tol": 0.01, "function": 'AVERAGE, MEDIAN, COUNTIF, SKEW',
     "working": "AVERAGE ≈ $73.77; MEDIAN ≈ $56.02; 304 of 480 orders (63%) are below the mean, pulled up by a long right tail (SKEW ≈ 3.04). The median is the typical order; the mean is right for totals (480 × mean = the quarter's revenue)."},
    {"id": 'P4.3', "tier": 4, "group": 'Case study',
     "q": "Delivery days by carrier: give each carrier's mean, sample standard deviation and coefficient of variation (Corvey first; two decimals, CV as a whole percent).",
     "parts": ['Corvey: mean days', 'Corvey: standard deviation (days)', 'Corvey: CV (%, whole number)', 'Tamsby: mean days', 'Tamsby: standard deviation (days)', 'Tamsby: CV (%, whole number)'],
     "answer": [2.97, 1.8, 61, 3.52, 0.87, 25], "tol": 0.01, "function": "AVERAGEIF; STDEV.S on each carrier's rows (the tab is sorted by carrier)",
     "working": 'Corvey: 2.97 days, s 1.80, CV ≈ 61%. Tamsby: 3.52 days, s 0.87, CV ≈ 25%. Corvey is quicker on average but more than twice as variable.'},
    {"id": 'P4.4', "tier": 4, "group": 'Case study',
     "q": 'For each carrier, give the five-number summary, the outlier limits (Q1 − 1.5 IQR, Q3 + 1.5 IQR), the number of outliers, and the number of late orders (more than five days).',
     "parts": ['Corvey: minimum', 'Corvey: Q1', 'Corvey: median', 'Corvey: Q3', 'Corvey: maximum', 'Corvey: lower limit', 'Corvey: upper limit', 'Corvey: outliers (count)', 'Corvey: late orders (count)', 'Tamsby: minimum', 'Tamsby: Q1', 'Tamsby: median', 'Tamsby: Q3', 'Tamsby: maximum', 'Tamsby: lower limit', 'Tamsby: upper limit', 'Tamsby: outliers (count)', 'Tamsby: late orders (count)'],
     "answer": [1.0, 2.0, 2.0, 3.0, 10.0, 0.5, 4.5, 25, 23, 1.0, 3.0, 3.0, 4.0, 6.0, 1.5, 5.5, 2, 1], "tol": 0, "function": 'QUARTILE.EXC, MIN, MAX, COUNTIFS',
     "working": 'Corvey: 1, 2, 2, 3, 10; limits 0.5 and 4.5; 25 outliers (every order of 5 days or more); 23 late (10.1%). Tamsby: 1, 3, 3, 4, 6; limits 1.5 and 5.5; 2 outliers (one 1-day, one 6-day order); 1 late (0.4%).'},
    {"id": 'P4.5', "tier": 4, "group": 'Case study',
     "q": "A fair comparison: Corvey carries a larger share of rural orders, which take longer for anyone. Give each carrier's late rate (%) within each zone (city, town, rural; one decimal), Corvey first. Does Corvey's higher late rate survive?",
     "parts": ['Corvey, city: late rate (%)', 'Corvey, town: late rate (%)', 'Corvey, rural: late rate (%)', 'Tamsby, city: late rate (%)', 'Tamsby, town: late rate (%)', 'Tamsby, rural: late rate (%)', "does Corvey's higher late rate survive? type one word: yes or no"],
     "answer": [7.9, 11.4, 14.3, 0.0, 0.0, 3.1, 'yes'], "tol": 0.05, "function": 'COUNTIFS',
     "working": "COUNTIFS by carrier, zone and days > 5, over the zone's orders. Corvey: 7.9%, 11.4%, 14.3%; Tamsby: 0%, 0%, 3.1%. Corvey is later in every zone, so the zone mix doesn't explain it."},
    {"id": 'P4.6', "tier": 4, "group": 'Case study',
     "q": "A second question: does the return rate differ by sales channel? Build the crosstab of channel × returned and give each channel's return rate (%; website, app, marketplace; one decimal).",
     "parts": ['website: return rate (%)', 'app: return rate (%)', 'marketplace: return rate (%)'],
     "answer": [6.3, 4.3, 4.0], "tol": 0.05, "function": 'COUNTIFS',
     "working": "COUNTIFS for each channel and returned = yes, divided by the channel's orders: website 15 / 239 = 6.3%, app 6 / 140 = 4.3%, marketplace 4 / 101 = 4.0%. The gaps are small next to the late-versus-on-time gap (20.8% against 4.4%)."},
    {"id": 'P4.7', "tier": 4, "group": 'Case study',
     "q": 'Marketing points to the correlation between discount and order value as proof that discounts make people spend more. Give r for all 480 orders, then r for the orders under $100 only (two decimals). What explains the difference?',
     "parts": ['r, all 480 orders (two decimals)', 'r, orders under $100 (two decimals)'],
     "answer": [0.65, 0.01], "tol": 0.01, "function": 'CORREL (sort by value, then use the rows under $100)',
     "working": "CORREL on all orders ≈ 0.65; on the 375 orders under $100 ≈ 0.01. The store's own rule gives 10% off orders of $100 or more, so big orders get discounts, not the other way round. To learn whether discounts raise spending, randomise them (an experiment)."},
    {"id": 'P4.8', "tier": 4, "group": 'Case study',
     "q": "Write the recommendation in one sentence with its numeric anchor: if all of next quarter's 480 orders went to one carrier at this quarter's late rates, about how many would break the five-day promise with each (Corvey first, whole numbers)?",
     "parts": ['late orders with Corvey (whole number)', 'late orders with Tamsby (whole number)'],
     "answer": [48.0, 2.0], "tol": 0, "function": 'COUNTIFS, then scale',
     "working": '480 × 23 / 228 ≈ 48; 480 × 1 / 252 ≈ 2. "Keep Tamsby: about half a day slower on average, but about 2 late orders in 480 instead of about 48, and late orders were returned at nearly five times the on-time rate."'},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch", 4: "Case"}
assert len(EX) == 35 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [9, 10, 8, 8]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g, w, q["tol"]) for g, w in zip(got, q["answer"])]
            if all(ok): print("✓ right" if n == 1 else f"✓ all {n} right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            if q["function"] not in ("—", ""): print(textwrap.fill(f"In a spreadsheet: {q['function']}.", 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers practice.py gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

## Tier 1 · Warm-up

One idea, one step, small numbers. The scales of measurement, a relative frequency, a pie's angle, a rate, the mean,
the median, the range, a z-score and the sign of a covariance.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

### How you'd do this in Python

Python's own `statistics` module has the measures of location; `numpy` does the same on arrays. A z-score is just
(x − mean) / s. Each line below computes one warm-up answer and checks it against the practice set.

In [ ]:
orders = [42, 58, 35, 61, 54]                        # P1.5
lead = [4, 9, 6, 5, 12, 7, 6]                         # P1.6
sales = [120, 95, 143, 110, 132]                      # P1.7

print("mean    ", st.mean(orders), "  numpy:", np.mean(orders))
print("median  ", st.median(lead), "   mode:", st.mode(lead))
print("range   ", max(sales) - min(sales), "  numpy:", np.ptp(sales))
print("z       ", (82 - 70) / 8)
print("rel. freq.", 18 / 50, "   conversion %", 22 / 400 * 100, "   pie angle", 0.25 * 360)

assert [st.mean(orders)] == answers("P1.5") and [st.median(lead)] == answers("P1.6")
assert [max(sales) - min(sales)] == answers("P1.7") and [(82 - 70) / 8] == answers("P1.8")
assert [18 / 50] == answers("P1.2") and [22 / 400 * 100] == answers("P1.4") and [0.25 * 360] == answers("P1.3")

### Move one value: the mean follows, the median stays

The course's twenty shipment times (hours, order to delivery). The slider changes the slowest one, 47 hours. Drag it
to 100: the mean (sky) moves from 30.65 to 33.3, while the median (gold) stays at 29.5. One extreme value pulls the
mean; the median doesn't care how far out it is.

In [ ]:
SHIP = [29, 34, 22, 38, 26, 31, 47, 27, 33, 24, 34, 29, 41, 25, 30, 26, 34, 23, 32, 28]

def mean_vs_median(slowest=47):
    xs = [slowest if x == 47 else x for x in SHIP]
    m, md = st.mean(xs), st.median(xs)
    fig, ax = plt.subplots(figsize=(11.5, 3.6))
    fig.subplots_adjust(left=0.04, right=0.98, top=0.78, bottom=0.2)
    dot_plot(ax, xs, color=CHALK)
    ax.axvline(m, color=SKY, lw=2.4); ax.axvline(md, color=GOLD, lw=2.4, ls="--")
    ax.text(m, 1.05, f"mean {m:.2f}", color=SKY, ha="left", va="bottom", fontsize=13, transform=ax.get_xaxis_transform())
    ax.text(md, 1.05, f"median {md:g}", color=GOLD, ha="right", va="bottom", fontsize=13, transform=ax.get_xaxis_transform())
    ax.set_xlim(18, 125); ax.set_ylim(-0.3, 1.0); ax.set_yticks([]); ax.set_xlabel("hours")
    fig.suptitle(f"Twenty shipment times, the slowest at {slowest} hours", x=0.04, ha="left", fontsize=15)
    footnote(fig); plt.show()

assert st.mean(SHIP) == 30.65 and st.median(SHIP) == 29.5
assert round(st.mean([100 if x == 47 else x for x in SHIP]), 2) == 33.3                    # the video's 47 → 100

In [ ]:
interact(mean_vs_median, slowest=IntSlider(value=47, min=20, max=120, step=1, description="slowest (h)"));

## Tier 2 · Core

The course's standard problems, several steps each: a sample against its population, a frequency distribution, row
percents, the weighted and geometric means, quartiles, the variance and standard deviation, the coefficient of
variation, a five-number summary with its outlier limit, and the covariance and correlation.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

**Quartiles.** The course places the pth percentile at position (p/100)(n + 1), with interpolation, which is a
spreadsheet's PERCENTILE.EXC and QUARTILE.EXC. In Python that's `statistics.quantiles(…, method="exclusive")` (its
default), or `numpy.percentile(…, method="weibull")`. The spreadsheet's PERCENTILE.INC is `method="inclusive"` in
`statistics` and numpy's default `method="linear"`. Say which one you used: on small data sets they can differ.

**Spread.** `statistics.variance` and `stdev` divide by n − 1 (a sample: VAR.S, STDEV.S); `pvariance` and `pstdev`
divide by n (a whole population: VAR.P, STDEV.P). numpy's `var` and `std` divide by n unless you pass `ddof=1`.

**Two variables.** `statistics.covariance` and `statistics.correlation` (Python 3.10 and later), or `numpy.cov` and
`numpy.corrcoef`. A weighted mean is `numpy.average(values, weights=…)`; a geometric mean `statistics.geometric_mean`.

In [ ]:
calls = [3, 5, 6, 7, 8, 8, 9, 10, 11, 12, 13, 15, 18, 21, 30]                          # P2.6
q1, q2, q3 = st.quantiles(calls, n=4, method="exclusive")
print("quartiles, exclusive (n + 1):", q1, q2, q3, "   numpy weibull:", np.percentile(calls, [25, 50, 75], method="weibull"))
print("quartiles, inclusive        :", st.quantiles(calls, n=4, method="inclusive"))
assert [q1, q2, q3] == answers("P2.6") and list(np.percentile(calls, [25, 50, 75], method="weibull")) == [q1, q2, q3]

returns = [1.2, -0.4, 0.8, 2.0, -1.1, 0.5]                                               # P2.7
print("s² (n − 1):", round(st.variance(returns), 4), "  s:", round(st.stdev(returns), 4),
      "  σ² (n):", round(st.pvariance(returns), 4), "  numpy std, ddof=1:", round(np.std(returns, ddof=1), 4))
assert abs(st.variance(returns) - answers("P2.7")[0]) < 1e-9 and round(st.stdev(returns), 2) == answers("P2.7")[1]

print("weighted mean:", np.average([15, 22, 40], weights=[120, 80, 50]))                  # P2.4
growth = st.geometric_mean([1.08, 1.12, 0.95])                                            # P2.5
print("average growth:", round(100 * (growth - 1), 2), "%")
assert abs(np.average([15, 22, 40], weights=[120, 80, 50]) - answers("P2.4")[0]) < 1e-9
assert round(100 * (growth - 1), 2) == answers("P2.5")[0]

minutes = [12, 17, 22, 14, 19, 26, 31, 18, 15, 23, 20, 16, 13, 27, 21, 18, 34, 16, 19, 24, 11, 17, 22, 14]   # P2.2
counts, edges = np.histogram(minutes, bins=[10, 15, 20, 25, 30, 35])                      # classes 10–14, 15–19, …
print("frequencies:", counts.tolist(), "  under 25 minutes:", round(100 * counts[:3].sum() / len(minutes), 2), "%")
assert counts.tolist() == answers("P2.2")[:5]

spend, sales2 = [2, 3, 5, 6, 9], [30, 34, 41, 45, 50]                                     # P2.10
print("covariance:", st.covariance(spend, sales2), "  r:", round(st.correlation(spend, sales2), 4),
      "  numpy:", np.cov(spend, sales2)[0, 1], round(np.corrcoef(spend, sales2)[0, 1], 4))
assert st.covariance(spend, sales2) == answers("P2.10")[0] and round(st.correlation(spend, sales2), 2) == answers("P2.10")[1]

In [ ]:
signups = [42, 38, 45, 51, 40, 44, 39, 47, 43, 95, 41]                                   # P2.9: the boxplot
q1, med, q3 = st.quantiles(signups, n=4, method="exclusive")
lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
assert [min(signups), q1, med, q3, max(signups), hi] == answers("P2.9")[:6]
fig, ax = plt.subplots(figsize=(11.5, 3.2))
fig.subplots_adjust(left=0.04, right=0.98, top=0.8, bottom=0.22)
inside = [x for x in signups if lo <= x <= hi]
ax.bxp([{"med": med, "q1": q1, "q3": q3, "whislo": min(inside), "whishi": max(inside),
         "fliers": [x for x in signups if not lo <= x <= hi]}], vert=False, widths=0.5, patch_artist=True,
       boxprops={"facecolor": SLATE, "edgecolor": CHALK, "linewidth": 2}, medianprops={"color": GOLD, "linewidth": 3},
       whiskerprops={"color": CHALK, "linewidth": 2}, capprops={"color": CHALK, "linewidth": 2},
       flierprops={"marker": "o", "markerfacecolor": CORAL, "markeredgecolor": CORAL, "markersize": 10})
ax.axvline(hi, color=CORAL, ls=":", lw=1.6); ax.text(hi, 1.35, f"limit {hi:g}", color=CORAL, ha="center")
ax.set_yticks([]); ax.set_xlabel("sign-ups a day")
fig.suptitle(f"Five numbers: {min(signups)} · {q1:g} · {med:g} · {q3:g} · {max(signups)}; 95 is beyond the limit",
             x=0.04, ha="left", fontsize=15)
footnote(fig); plt.show()

### The standard deviation as a band

The course's weekly defect counts, 2, 4, 4, 5 and 10: mean 5, s = 3. The slider moves the last week. The violet
band is x̄ ± s. Bring the 10 in towards the others and watch s shrink; push it out and the band widens, because every
squared deviation counts.

In [ ]:
def sd_band(last=10):
    xs = [2, 4, 4, 5, last]
    m, s = st.mean(xs), st.stdev(xs)
    fig, ax = plt.subplots(figsize=(11.5, 3.4))
    fig.subplots_adjust(left=0.04, right=0.98, top=0.78, bottom=0.2)
    ax.axvspan(m - s, m + s, color=VIOLET, alpha=0.25, lw=0)
    dot_plot(ax, xs, color=CHALK, size=90)
    ax.axvline(m, color=SKY, lw=2.4)
    ax.text(m, 0.92, f"x̄ = {m:g}", color=SKY, ha="center", fontsize=13)
    ax.set_xlim(-6, 26); ax.set_ylim(-0.3, 1.1); ax.set_yticks([]); ax.set_xlabel("defects a week")
    fig.suptitle(f"s² = Σ(x − x̄)² ÷ (n − 1) = {sum((x - m) ** 2 for x in xs):g} ÷ 4 = {st.variance(xs):g};   s = {s:.2f}",
                 x=0.04, ha="left", fontsize=15, color=VIOLET)
    footnote(fig); plt.show()

assert st.variance([2, 4, 4, 5, 10]) == 9 and st.stdev([2, 4, 4, 5, 10]) == 3              # the video's 9 and 3

In [ ]:
interact(sd_band, last=IntSlider(value=10, min=0, max=20, description="last week"));

## Tier 3 · Stretch

Combine two ideas, choose the method, spot the trap, work backwards: Simpson's paradox, a truncated axis, the mean of
growth rates, a missing value from its mean, what a shift and a scale do to s, Chebyshev and the empirical rule, the
two outlier rules disagreeing, and a correlation of zero that hides a strong relationship.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

`pandas` puts a data set in a `Series` (one column) or a `DataFrame` (a table). `describe()` gives the count, mean,
standard deviation (n − 1), quartiles and extremes in one go; note that pandas' quartiles use the inclusive rule
(PERCENTILE.INC), so for the course's rule use `statistics.quantiles` as above. z-scores are one line on a Series.
A crosstab with row percents is `pd.crosstab(…, normalize="index")`.

In [ ]:
times = pd.Series([20, 22, 23, 23, 24, 25, 25, 26, 27, 40], name="minutes")              # P3.7
print(times.describe().round(3).to_string())
z = (times - times.mean()) / times.std()
q1, _, q3 = st.quantiles(times, n=4, method="exclusive")
print("\nz of 40:", round(z.iloc[-1], 2), "  quartile rule's upper limit:", q3 + 1.5 * (q3 - q1))
assert [q3 + 1.5 * (q3 - q1), round(z.iloc[-1], 2)] == answers("P3.7")

reps = pd.DataFrame({"rep": ["Rhea"] * 100 + ["Tomas"] * 100,                             # P3.1, one row per deal
                     "size": ["small"] * 50 + ["large"] * 50 + ["small"] * 90 + ["large"] * 10,
                     "won": [1] * 40 + [0] * 10 + [1] * 15 + [0] * 35 + [1] * 63 + [0] * 27 + [1] * 2 + [0] * 8})
print("\noverall close rate (%):\n", (100 * reps.groupby("rep")["won"].mean()).to_string())
print("\nwithin each deal size (%):\n", (100 * reps.pivot_table(index="rep", columns="size", values="won")).round(1).to_string())
assert (100 * reps.groupby("rep")["won"].mean()).round(9).tolist() == answers("P3.1")[:2]

fives = [8, 10, 16, 12]; fifth = 5 * 12 - sum(fives)                                     # P3.4: work backwards
assert [fifth, round(st.stdev(fives + [fifth]), 2)] == answers("P3.4")
base = np.array([20, 24, 25, 31, 30])                                                   # P3.5: shift and scale
print("\ns:", round(base.std(ddof=1), 4), " +5:", round((base + 5).std(ddof=1), 4), " ×1.1:", round((1.1 * base).std(ddof=1), 4))

In [ ]:
price, profit = [8, 9, 10, 11, 12], [30, 45, 50, 45, 30]                                 # P3.8: r = 0, and yet
r = st.correlation(price, profit)
assert abs(r - answers("P3.8")[0]) < 1e-9
fig, ax = plt.subplots(figsize=(8, 4.2))
fig.subplots_adjust(left=0.1, right=0.97, top=0.85, bottom=0.16)
ax.plot(price, profit, color=GHOST, lw=1.5); ax.scatter(price, profit, s=90, color=CHALK, zorder=3)
ax.set_xlabel(r"price (\$)"); ax.set_ylabel(r"weekly profit (\$k)")
fig.suptitle(f"r = {abs(r):.2f}, yet price matters: an arch peaking at " + r"\$10", x=0.1, ha="left", fontsize=15, color=GOLD)
footnote(fig); plt.show()

### One point can move r

The course's six days of returns (%): the market and one stock. r ≈ 0.96. The slider adds a seventh day: the market
falls 0.1%, and the stock moves by the slider's amount (company news). At +6.0%, r falls to about 0.52.

In [ ]:
MKT = [-1.3, -0.9, -0.1, 0.3, 0.7, 1.9]
STK = [-2.4, -0.8, -0.4, 1.2, 0.8, 2.8]

def seventh_day(stock=6.0):
    xs, ys = MKT + [-0.1], STK + [stock]
    r6, r7 = st.correlation(MKT, STK), st.correlation(xs, ys)
    fig, ax = plt.subplots(figsize=(8.5, 5))
    fig.subplots_adjust(left=0.1, right=0.97, top=0.86, bottom=0.14)
    ax.axvline(np.mean(xs), color=SKY, lw=1.4, alpha=0.7); ax.axhline(np.mean(ys), color=SKY, lw=1.4, alpha=0.7)
    ax.scatter(MKT, STK, s=90, color=CHALK, zorder=3); ax.scatter([-0.1], [stock], s=130, color=CORAL, zorder=4)
    ax.set_xlim(-2, 2.5); ax.set_ylim(-4, 8); ax.set_xlabel("market (%)"); ax.set_ylabel("stock (%)")
    fig.suptitle(f"six days: r = {r6:.2f}   ·   with the seventh: r = {r7:.2f}", x=0.1, ha="left", fontsize=15, color=GOLD)
    footnote(fig); plt.show()

assert round(st.covariance(MKT, STK), 9) == 2 and round(st.correlation(MKT, STK), 2) == 0.96
assert round(st.correlation(MKT + [-0.1], STK + [6.0]), 2) == 0.52                        # the video's 0.96 → 0.52

In [ ]:
interact(seventh_day, stock=FloatSlider(value=6.0, min=-3.0, max=7.0, step=0.1, readout_format=".1f",
                                        description="stock (%)"));

## Tier 4 · The case: Wrenfield Home's carrier decision

Wrenfield Home sells homeware online (a fictional company; our own data). Last quarter it used two delivery
carriers, Corvey and Tamsby, and it wants to keep one. Its promise: delivery within five days. Here are the quarter's
480 orders, one row each:

| column | what it is |
|---|---|
| `order` | order number, 1–480 |
| `zone` | delivery zone: city, town or rural |
| `channel` | where it was ordered: website, app or marketplace |
| `carrier` | Corvey or Tamsby |
| `value` | order value before discount, $ |
| `items` | items in the order |
| `discount` | % off: 10 on orders of $100 or more (the store's rule), plus 5 with a promotional code |
| `days` | days from order to delivery (late when more than 5) |
| `returned` | yes or no |

The next cell builds the table from a seeded random generator, so everyone gets exactly the same 480 orders (the
same as the spreadsheet's Case study tab), and checks a few of its numbers.

In [ ]:
CASE_SEED, CASE_N, PROMISE_DAYS = 1634, 480, 5

def build_case(seed=CASE_SEED, n=CASE_N):
    rng = random.Random(seed); u = rng.random
    def z():                                                   # a standard normal draw (Box–Muller, from random())
        a, b = u(), u()
        return math.sqrt(-2 * math.log(1 - a)) * math.cos(2 * math.pi * b)
    rows = []
    for i in range(n):
        a = u(); zone = "city" if a < 0.55 else "town" if a < 0.85 else "rural"
        a = u(); channel = "website" if a < 0.5 else "app" if a < 0.8 else "marketplace"
        carrier = "Corvey" if u() < {"city": 0.44, "town": 0.5, "rural": 0.66}[zone] else "Tamsby"
        cents = round(math.exp(math.log(55) + 0.7 * z()) * 100)            # right-skewed order values
        items = max(1, min(9, round(cents / 2800 + 0.6 * z())))
        discount = (10 if cents >= 10000 else 0) + (5 if u() < 0.25 else 0)
        if carrier == "Tamsby":                                             # steady
            d = {"city": 3.1, "town": 3.7, "rural": 4.5}[zone] + 0.75 * z()
        else:                                                               # quicker, but some parcels wait at a depot
            d = {"city": 2.1, "town": 2.7, "rural": 3.5}[zone] + 0.6 * z()
            if u() < 0.14: d += 3 + 4 * u()
        days = max(1, round(d))
        returned = u() < 0.04 + 0.2 * (days > PROMISE_DAYS)
        rows.append((i + 1, zone, channel, carrier, cents / 100, items, discount, days, "yes" if returned else "no"))
    return pd.DataFrame(rows, columns=["order", "zone", "channel", "carrier", "value", "items", "discount", "days",
                                       "returned"])

df = build_case()

# The case's numbers as the course computed them, to check this table against.
# @STATS
CASE_STATS = {'n': 480,
 'zone_n': {'city': 246, 'town': 167, 'rural': 67},
 'channel_n': {'website': 239, 'app': 140, 'marketplace': 101},
 'carrier_n': {'Corvey': 228, 'Tamsby': 252},
 'value_mean': 73.7664791667,
 'value_median': 56.02,
 'value_sd': 59.6471772751,
 'value_skew': 3.042310862,
 'value_min': 7.11,
 'value_max': 620.95,
 'value_total': 35407.91,
 'value_below_mean': 304,
 'value_freq': [192, 183, 64, 25, 8, 3, 3, 1, 0, 0, 0, 0, 1],
 'days_mean': {'Corvey': 2.9692982456, 'Tamsby': 3.5198412698},
 'days_median': {'Corvey': 2.0, 'Tamsby': 3.0},
 'days_sd': {'Corvey': 1.7991494414, 'Tamsby': 0.867521068},
 'days_cv': {'Corvey': 60.5917389416, 'Tamsby': 24.6465962944},
 'days_five': {'Corvey': (1.0, 2.0, 2.0, 3.0, 10.0), 'Tamsby': (1.0, 3.0, 3.0, 4.0, 6.0)},
 'days_fences': {'Corvey': (0.5, 4.5), 'Tamsby': (1.5, 5.5)},
 'days_outliers_n': {'Corvey': 25, 'Tamsby': 2},
 'late_n': {'Corvey': 23, 'Tamsby': 1},
 'late_by_zone': {'Corvey': {'city': (9, 114), 'town': (9, 79), 'rural': (5, 35)},
                  'Tamsby': {'city': (0, 132), 'town': (0, 88), 'rural': (1, 32)}},
 'zone_by_carrier': {'Corvey': {'city': 114, 'town': 79, 'rural': 35},
                     'Tamsby': {'city': 132, 'town': 88, 'rural': 32}},
 'returns_late': (5, 24),
 'returns_ontime': (20, 456),
 'returns_carrier': {'Corvey': (18, 228), 'Tamsby': (7, 252)},
 'returns_channel': {'website': (15, 239), 'app': (6, 140), 'marketplace': (4, 101)},
 'disc_value_cov': 177.4964939979,
 'disc_value_r': 0.6542924384,
 'disc_value_r_under100': 0.0141831087,
 'under100_n': 375}
# @END STATS
S = CASE_STATS
assert len(df) == S["n"] and df["zone"].value_counts().to_dict() == S["zone_n"]
assert df["channel"].value_counts().to_dict() == S["channel_n"] and df["carrier"].value_counts().to_dict() == S["carrier_n"]
assert abs(df["value"].mean() - S["value_mean"]) < 1e-9 and abs(df["value"].median() - S["value_median"]) < 1e-9
assert abs(df["value"].sum() - S["value_total"]) < 1e-6 and (df["value"] < df["value"].mean()).sum() == S["value_below_mean"]
assert abs(df["value"].skew() - S["value_skew"]) < 1e-9                                  # pandas' skew is SKEW's formula
assert np.histogram(df["value"], bins=range(0, 700, 50))[0].tolist() == S["value_freq"]
for c in ("Corvey", "Tamsby"):
    d = df.loc[df["carrier"] == c, "days"]
    assert abs(d.mean() - S["days_mean"][c]) < 1e-9 and abs(d.std() - S["days_sd"][c]) < 1e-9
    assert (d > PROMISE_DAYS).sum() == S["late_n"][c]
assert abs(df["discount"].corr(df["value"]) - S["disc_value_r"]) < 1e-9
print(f"{len(df)} orders, {df.shape[1]} columns: the same data as the course's case.")
df.head(8)

### How you'd do this in Python

With the orders in a DataFrame, each step of the case is a line or two: `value_counts(normalize=True)` for a
frequency table in percents; `describe()` and `skew()` for one variable; `groupby("carrier")` to compare the carriers;
`pd.crosstab` for two categorical variables, with `normalize="index"` for row percents; `corr()` for r; and
`boxplot` for the two carriers side by side. These cells reproduce the video's case numbers. Then it's your turn.

In [ ]:
print("Zones (%):\n", (100 * df["zone"].value_counts(normalize=True)).round(2).to_string(), "\n")
print("Order value ($):\n", df["value"].describe().round(2).to_string(), "\n  skew", round(df["value"].skew(), 2), "\n")
print("Delivery days by carrier:\n",
      df.groupby("carrier")["days"].agg(["count", "mean", "median", "std"]).round(2).to_string(), "\n")
late = df["days"] > PROMISE_DAYS
print("Returned, by late or on time (row %):\n",
      (100 * pd.crosstab(late.map({True: "late", False: "on time"}), df["returned"], normalize="index")).round(1).to_string())

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.6), gridspec_kw={"width_ratios": [1.25, 1]})
fig.subplots_adjust(left=0.06, right=0.98, top=0.82, bottom=0.15, wspace=0.25)
ax1.hist(df["value"], bins=range(0, 700, 50), color=CHALK, edgecolor=SLATE)
ax1.axvline(df["value"].mean(), color=SKY, lw=2.4); ax1.axvline(df["value"].median(), color=GOLD, lw=2.4, ls="--")
ax1.set_title(rf"order value: mean \${df['value'].mean():.2f} (sky), median \${df['value'].median():.2f} (gold)",
              loc="left", fontsize=13)
ax1.set_xlabel(r"order value (\$)"); ax1.set_ylabel("orders")
groups = [df.loc[df["carrier"] == c, "days"] for c in ("Corvey", "Tamsby")]
bp = ax2.boxplot(groups, vert=False, whis=1.5, widths=0.5, patch_artist=True,
                 medianprops={"color": GOLD, "linewidth": 3}, flierprops={"marker": "o", "markersize": 6,
                 "markerfacecolor": CORAL, "markeredgecolor": CORAL})
for box, c in zip(bp["boxes"], ("Corvey", "Tamsby")): box.set(facecolor=SLATE, edgecolor=PAIR[c], linewidth=2)
for k, c in enumerate(("Corvey", "Tamsby")):
    for part in ("whiskers", "caps"):
        for line in bp[part][2 * k: 2 * k + 2]: line.set(color=PAIR[c], linewidth=2)
ax2.axvline(PROMISE_DAYS + 0.5, color=CORAL, ls=":", lw=1.6)
ax2.set_yticks([1, 2], ["Corvey", "Tamsby"]); ax2.set_xlabel("days to delivery")
ax2.set_title("delivery days (both quartile rules agree here)", loc="left", fontsize=13)
fig.suptitle("Wrenfield Home · one quarter, 480 orders", x=0.06, ha="left", fontsize=15)
footnote(fig, "a fictional company · our own data · computed"); plt.show()

### Your turn: tier 4

Reproduce the video's case numbers, then go further: returns by sales channel, the discount claim checked yourself,
and the recommendation with its number. Use the DataFrame `df` in a cell of your own (press **+** to add one), then
type your answers.

In [ ]:
practice("P4.1")

In [ ]:
practice("P4.2")

In [ ]:
practice("P4.3")

In [ ]:
practice("P4.4")

In [ ]:
practice("P4.5")

In [ ]:
practice("P4.6")

In [ ]:
practice("P4.7")

In [ ]:
practice("P4.8")

### The class width changes the picture

The order values again, with a slider for the class width. Narrow classes show noise; very wide ones hide the long
right tail. Somewhere between, the shape is clear: most orders between $20 and $100, a few far out to the right.

In [ ]:
def value_histogram(width=50):
    fig, ax = plt.subplots(figsize=(11.5, 4.2))
    fig.subplots_adjust(left=0.07, right=0.98, top=0.84, bottom=0.15)
    edges = np.arange(0, df["value"].max() + width, width)
    counts, _, _ = ax.hist(df["value"], bins=edges, color=CHALK, edgecolor=SLATE)
    ax.set_xlabel(r"order value (\$)"); ax.set_ylabel("orders")
    fig.suptitle(rf"classes of \${width}: {len(edges) - 1} classes, the tallest holds {int(counts.max())} orders",
                 x=0.07, ha="left", fontsize=15)
    footnote(fig, "a fictional company · our own data · computed"); plt.show()

In [ ]:
interact(value_histogram, width=IntSlider(value=50, min=5, max=200, step=5, description="width ($)"));

### The tier 4 answers, from the DataFrame

Try the questions first. This cell works every tier 4 answer out from `df` with pandas and checks it against the
practice set: open it when you've finished, or when you're stuck.

In [ ]:
ZONES, CHANNELS, CARRIERS = ("city", "town", "rural"), ("website", "app", "marketplace"), ("Corvey", "Tamsby")
def pct(part, whole): return 100 * part / whole
def five(d):
    q1, med, q3 = st.quantiles(d, n=4, method="exclusive")
    return [min(d), q1, med, q3, max(d)], (q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1))

mine = {}
mine["P4.1"] = [round(pct((df["zone"] == z_).sum(), len(df)), 2) for z_ in ZONES]
mine["P4.2"] = [round(df["value"].mean(), 2), round(df["value"].median(), 2),
                round(pct((df["value"] < df["value"].mean()).sum(), len(df))), "median"]
by = {c: df.loc[df["carrier"] == c, "days"].tolist() for c in CARRIERS}
mine["P4.3"] = [v for c in CARRIERS for v in (round(st.mean(by[c]), 2), round(st.stdev(by[c]), 2),
                                              round(100 * st.stdev(by[c]) / st.mean(by[c])))]
mine["P4.4"] = []
for c in CARRIERS:
    f5, (lo, hi) = five(by[c])
    mine["P4.4"] += f5 + [lo, hi, sum(1 for x in by[c] if not lo <= x <= hi), sum(1 for x in by[c] if x > PROMISE_DAYS)]
late_rate = df.assign(late=df["days"] > PROMISE_DAYS).pivot_table(index="carrier", columns="zone", values="late", aggfunc="mean")
mine["P4.5"] = [round(100 * late_rate.loc[c, z_], 1) for c in CARRIERS for z_ in ZONES] + ["yes"]
ret = pd.crosstab(df["channel"], df["returned"], normalize="index")
mine["P4.6"] = [round(100 * ret.loc[c, "yes"], 1) for c in CHANNELS]
under = df[df["value"] < 100]
mine["P4.7"] = [round(df["discount"].corr(df["value"]), 2), round(under["discount"].corr(under["value"]), 2)]
mine["P4.8"] = [round(len(df) * pct(sum(1 for x in by[c] if x > PROMISE_DAYS), len(by[c])) / 100) for c in CARRIERS]

for k, got in mine.items():
    ok = all(right(g if isinstance(g, str) else float(g), w, Q[k]["tol"]) for g, w in zip(got, answers(k)))
    assert ok and len(got) == len(answers(k)), (k, got, answers(k))
    print(f"{k}: " + " · ".join(f"{p}: {show(float(g) if not isinstance(g, str) else g)}" for p, g in zip(Q[k]["parts"], got)))
print("\nEvery tier 4 answer worked out from the DataFrame matches the practice set.")

## Sources
Every exercise, data set and company here is our own (Wrenfield Home, Corvey and Tamsby are fictional). The chapter
order follows a standard first course in business statistics (data, tables and charts, numerical measures); the
definitions agree with OpenStax, Introductory Business Statistics 2e and Introductory Statistics 2e (retrieved
2026-11-06).

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib
_src = pathlib.Path("../../../episodes/bs01_describing_data/practice.py")
if _src.exists():
    _spec = importlib.util.spec_from_file_location("bs01_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["tol"], _q["function"]) == (_e["q"], _e["working"], _e["tol"], _e["function"]), _e["id"]
        assert all((a == b) if isinstance(b, str) else abs(a - float(b)) < 1e-9
                   for a, b in zip(_q["answer"], _flat(_e["answer"]))) and len(_q["answer"]) == len(_flat(_e["answer"])), _e["id"]
    assert len(_P.EX) == len(EX) and _P.CASE["days"] == df["days"].tolist() and _P.CASE["value"] == df["value"].tolist()
    print("matches the course's practice set")